# Foundry Scenarios

The Foundry scenario family provides the `RedTeamAgent` — a comprehensive red teaming scenario
that combines converter-based attacks (encoding/obfuscation), multi-turn attacks (Crescendo,
RedTeaming), and technique composition. It's organized into difficulty levels: EASY, MODERATE,
and DIFFICULT.

For full programming details, see
[Common Scenario Parameters](../code/scenarios/1_common_scenario_parameters.ipynb).

In [ ]:
from pathlib import Path

from pyrit.output import output_scenario_async
from pyrit.registry import TargetRegistry
from pyrit.scenario import DatasetAttackConfiguration, DatasetSource
from pyrit.scenario.foundry import FoundryTechnique, RedTeamAgent
from pyrit.setup import initialize_from_config_async

await initialize_from_config_async(config_path=Path("pyrit_conf.yaml"))  # type: ignore

objective_target = TargetRegistry.get_registry_singleton().instances.get("openai_chat")

Auto-discovered plaintext environment file ./.pyrit/.env will be loaded. Azure Key Vault through env_akv_ref is more secure for shared or deployed secrets; use .env.local only for deliberate local overrides. To inspect a resolved AKV-only configuration from a source checkout, run `python -m build_scripts.export_akv_environment`; it writes ~/.pyrit/.env_akv.


TextAdaptive: _EXCLUDED_TECHNIQUES entries ['prompt_sending'] are not in the current scenario-techniques catalog ['context_compliance', 'crescendo_history_lecture', 'crescendo_journalist_interview', 'crescendo_movie_director', 'crescendo_simulated', 'many_shot', 'pair', 'red_teaming', 'role_play', 'tap', 'violent_durian']; the exclusion is a no-op for those entries. Remove stale entries or update the catalog.


## RedTeamAgent

Tests a target using a wide range of attack techniques — from simple encoding converters to
complex multi-turn conversations. The default dataset is HarmBench.

**CLI example:**

```bash
pyrit_scan run foundry.red_team_agent --target openai_chat --techniques base64 --max-dataset-size 1
```

**Available techniques by difficulty:**

| Difficulty | Techniques |
|---|---|
| **EASY** | AnsiAttack, AsciiArt, AsciiSmuggler, Atbash, Base64, Binary, Caesar, CharacterSpace, CharSwap, Diacritic, Flip, Jailbreak, Leetspeak, Morse, ROT13, StringJoin, SuffixAppend, UnicodeConfusable, UnicodeSubstitution, Url |
| **MODERATE** | Tense |
| **DIFFICULT** | Crescendo, MultiTurn, Pair, Tap |
| **Aggregates** | ALL, EASY, MODERATE, DIFFICULT |

In [ ]:
dataset_config = DatasetAttackConfiguration(sources=[DatasetSource(name="harmbench")], max_total=1)

scenario = RedTeamAgent()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [FoundryTechnique.Base64],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

print(f"Scenario: {scenario.name}")
print(f"Atomic attacks: {scenario.atomic_attack_count}")

scenario_result = await scenario.run_async()  # type: ignore

Scenario: RedTeamAgent
Atomic attacks: 2


In [ ]:
await output_scenario_async(scenario_result)


                                  📊 SCENARIO RESULTS: RedTeamAgent                                  

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: RedTeamAgent
    • Result ID: 3a063d48-3251-4f29-af89-e45787c9d675
    • Scenario Version: 1
    • PyRIT Version: 1.2.0.dev0
    • Description:
        RedTeamAgent is a preconfigured scenario that automatically generates multiple AtomicAttack instances based on
        the specified attack techniques. It supports both single-turn attacks (with various converters) and multi-turn
        attacks (Crescendo, RedTeaming), making it easy to quickly test a target against multiple attack vectors. The
        scenario can expand difficulty levels (EASY, MODERATE, DIFFICULT) into their constituent attack techniques, or
        you can specify individual techniques directly. This scenario is designed for use with the Foundry AI Red
       

## Technique Composition

You can pair a multi-turn attack with one or more converter techniques using `FoundryComposite`.
Each converter in the composite is applied in sequence before the attack runs.

```python
from pyrit.scenario.foundry import FoundryComposite

composed = FoundryComposite(attack=FoundryTechnique.Crescendo, converters=[FoundryTechnique.Caesar, FoundryTechnique.CharSwap])
```

In [ ]:
# from pyrit.scenario.foundry import FoundryComposite
# composed = FoundryComposite(attack=FoundryTechnique.Crescendo, converters=[FoundryTechnique.Caesar, FoundryTechnique.CharSwap])
# scenario_techniques = [FoundryTechnique.Base64, composed]

For more details, see the [Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb) and
[Configuration](../getting_started/configuration.md).